# 03 — K-Means regimes and feature engineering

K-Means creates a secondary label for the HVAC operating regime. The clusterer is fitted on the temporal training set only; validation and test data are only transformed.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]

import matplotlib.pyplot as plt
import seaborn as sns

from cooling_load.clustering import OperatingRegimeClusterer
from cooling_load.config import load_config
from cooling_load.io import read_frame
from cooling_load.pipeline import run_feature_engineering
from cooling_load.splitting import temporal_holdout

config = load_config(ROOT / "configs/base.yaml")
clean = read_frame(config.path("interim_data"))
featured = run_feature_engineering(config, clean)
train, validation, test = temporal_holdout(featured, config.data.timestamp_column)

In [ ]:
clusterer = OperatingRegimeClusterer(config.clustering.features, config.clustering.n_clusters)
train_clustered = clusterer.fit_transform(train, "cooling_load_kwh_lag_3")
validation_clustered = clusterer.transform(validation)
display(train_clustered.groupby("operating_regime")[list(config.clustering.features)].median())

In [ ]:
sns.countplot(
    data=train_clustered,
    x="operating_regime",
    order=["low_load", "normal_load", "high_load", "peak_load"],
)
plt.xticks(rotation=25)
plt.title("Training operating regimes")
plt.show()
display(featured.filter(regex="lag_|mean_|std_|temperature_delta|thermal_load").head())
print("Feature dataset:", config.path("feature_data"))